In [1]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB")

Connected to DuckDB


In [2]:
# Verify listings_final exists
count = con.execute("SELECT COUNT(*) FROM listings_final").fetchone()[0]
print(f"Source data: {count:,} rows in listings_final")

Source data: 96,182 rows in listings_final


## Star Schema Design Decisions

### Why Star Schema?

**Chosen over:** Normalized (3NF) schema  
**Reasoning:**
- Optimized for analytical queries (OLAP)
- Simpler joins for reporting
- Industry standard for data warehousing
- The assignment explicitly requires it (Section 3.4)

### Dimension Table Keys

| Table | Grain | Surrogate Key | Natural Key |
|-------|-------|---------------|-------------|
| dim_host | One row per unique host | host_id (natural) | host_id |
| dim_location | One row per unique neighbourhood | location_id (surrogate) | neighbourhood_cleansed |
| dim_property | One row per (room_type + property_category) combo | property_id (surrogate) | room_type + category |

### Fact Table Keys

| Key | Type | References |
|-----|------|------------|
| listing_id | Natural PK | listings.id |
| host_id | FK | dim_host.host_id |
| location_id | FK (surrogate) | dim_location.location_id |
| property_id | FK (surrogate) | dim_property.property_id |

### Why Some Keys Are Surrogate?

- `location_id`: Neighbourhood names can change; surrogate key is stable
- `property_id`: Composite key (room_type + category) is cumbersome; surrogate simplifies joins

Creating dim_host

In [3]:
# one row per unique host

con.execute("""
    CREATE OR REPLACE TABLE dim_host AS
    SELECT DISTINCT
        host_id,
        host_name,
        host_since,
        host_location,
        host_is_superhost,
        superhost_status,
        host_response_rate,
        host_acceptance_rate,
        calculated_host_listings_count
    FROM listings_final
    WHERE host_id IS NOT NULL
    ORDER BY host_id
""")

host_count = con.execute("SELECT COUNT(*) FROM dim_host").fetchone()[0]
print(f"dim_host created: {host_count:,} unique hosts")

dim_host created: 57,023 unique hosts


Creating dim_location

In [4]:
# one row per unique neighbourhood

con.execute("""
    CREATE OR REPLACE TABLE dim_location AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY neighbourhood_cleansed) as location_id,
        neighbourhood_cleansed as neighbourhood,
        AVG(latitude) as avg_latitude,
        AVG(longitude) as avg_longitude,
        COUNT(*) as listing_count
    FROM listings_final
    WHERE neighbourhood_cleansed IS NOT NULL
    GROUP BY neighbourhood_cleansed
    ORDER BY neighbourhood_cleansed
""")

loc_count = con.execute("SELECT COUNT(*) FROM dim_location").fetchone()[0]
print(f" dim_location created: {loc_count} neighbourhoods")

# top neighborhoods by listing count
print("\n Top 10 neighborhoods by listing count:")
print(con.execute("""
    SELECT neighbourhood, listing_count
    FROM dim_location
    ORDER BY listing_count DESC
    LIMIT 10
""").df().to_string(index=False))

BinderException: Binder Error: Referenced column "neighbourhood_cleansed" not found in FROM clause!
Candidate bindings: "listings_final.neighbourhood_final", "listings_final.neighbourhood", "listings_final.host_location", "listings_final.review_scores_location", "listings_final.review_scores_value"
LINE 10:     WHERE neighbourhood_cleansed IS NOT NULL
                   ^

In [11]:
con.close()
print("Connection closed!")

Connection closed!
